In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import os
import json
os.chdir("..")

In [2]:
from datasets import load_dataset

In [3]:
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation

model_id = "Qwen/QwQ-32B"
tokenizer = initialize_tokenizer(model_id)

In [4]:
dataset = load_dataset("dmitriihook/blocksworld-6-blocks-qwq-reasoning-parts-low-v4")["train"]

(…)6-blocks-qwq-reasoning-parts-low-v4.json:   0%|          | 0.00/102k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20 [00:00<?, ? examples/s]

In [5]:
dataset_gen = load_dataset("dmitriihook/qwq-32b-planning-6-blocks")["train"]

In [10]:
idx = 10

print(
    dataset_gen[idx]["generation"]
)

Okay, let's see. I need to figure out the plan to rearrange the blocks from the initial state to the goal state. Let me start by understanding the problem step by step.

First, the initial conditions are:

- Block D is clear.
- Hand is empty.
- Block A is on top of Block B.
- Block B is on top of Block C.
- Block C is on top of Block F.
- Block D is on top of Block A.
- Block F is on top of Block E.
- Block E is on the table.

Wait, hold on. Let me parse that again. The initial conditions say:

- Block D is clear (so nothing is on top of D).
- Hand is empty.
- A is on B, which is on C, which is on F, which is on E (since F is on E, and E is on the table). Then D is on top of A. So the stack for A is: D is on A, which is on B, which is on C, F, E. Wait, but the way it's written: "Block D is on top of Block A" so the stack would be D on A, then A on B, etc. So the initial stacks are:

Starting from the table:

E is on the table. On E is F. On F is C. On C is B. On B is A. On A is D. And 

In [11]:
print(
    len(tokenizer.encode(dataset[idx]["label"])), len(tokenizer.encode(dataset_gen[idx]["generation"]))
)
print(dataset[idx]["label"])

2540 6392
["initial-state-understanding"]- Block D is clear.
- Hand is empty.
- Block A is on top of Block B.
- Block B is on top of Block C.
- Block C is on top of Block F.
- Block D is on top of Block A.
- Block F is on top of Block E.
- Block E is on the table.["end-section"]

["initial-state-understanding"]E is on the table. On E is F. On F is C. On C is B. On B is A. On A is D. And D is clear. So the stack is E -> F -> C -> B -> A -> D. And D is the top block here, clear. Also, the hand is empty.["end-section"]

["goal-state-understanding"]- A on D,

- B on C,

- D on F,

- E on A,

- F on B.["end-section"]

["goal-state-understanding"]1. A is on D.

2. B is on C.

3. D is on F.

4. E is on A.

5.  F is on B.["end-section"]

["goal-state-understanding"]- A is on D. So D must be under A.

- D is on F. So F is under D, which is under A.

- F is on B. So B is under F, which is under D, which is under A.["end-section"]

["goal-state-understanding"]C -> B -> F -> D -> A -> E.["end-sect

In [11]:
import re

labels_low = [
    "initial-state-understanding",
    "goal-state-understanding",
    "comparative-analysis",
    "recursive-search",
    "plan-formulation",
    "constraint-analysis",
    "stating-actions",
    "state-tracking"
]

In [12]:
def parse_sections(text:str, labels: list[str]) -> list[tuple[str, str]]:
    pattern = r'\["([^"]+)"\](.*?)\["end-section"\]'
    matches = re.findall(pattern, text, re.DOTALL)
    
    sections = []
    for label, content in matches:
        if label in labels:
            sections.append((label, content.strip()))
    
    return sections

parse_sections(dataset[idx]["label"], labels_low)

[('initial-state-understanding',
  "Okay, let's see. I need to figure out the plan to rearrange the blocks from the initial state to the goal state. Let me start by understanding the problem step by step.\n\nFirst, the initial conditions are:\n\n- Block A is clear (so nothing is on top of it)\n- Block E is clear\n- Block F is clear\n- Hand is empty\n- The stack structure is: A on B, B on C, C on D, and D is on the table. Then, E and F are both on the table."),
 ('goal-state-understanding',
  'The goal is to have:\n\n- B on top of A\n- C on top of B\n- D on top of C\n- E on top of D\n- F on top of E'),
 ('comparative-analysis',
  'Wait, so the final stack should be A at the bottom, then B, C, D, E, F on top? Or is it B on A, then C on B, etc., forming a vertical stack? Let me parse the goal correctly. The goal states each block is on top of the previous one in the sequence B -> A? Wait, no. Wait, the goal says "Block B is on top of Block A", so B is above A. Then "C is on top of B", so 

In [30]:
def map_sections_into_tokens(sections: list[tuple[str, str]], row: dict) -> list[dict]:
    # tokens = tokenize_blocksworld_generation(tokenizer, row)
    generation = row["generation"]
    section_tokens = []
    for label, content in sections[1:]:
        text_pos = generation.find(content[:300])
        if text_pos == -1:
            continue
        text_before = generation[:text_pos]
        tokens_before = tokenize_blocksworld_generation(tokenizer, row, text_before)[0, :-2]
        content_tokens = tokenizer.encode(" " + content)[:-5]
        section_tokens.append({
            "label": label,
            "pos_before": len(tokens_before),
            "pos_after": len(tokens_before) + len(content_tokens),
            "text_pos": text_pos,
            "content": content
        })

    return section_tokens

def test_sections():
    idx = 10
    generation = dataset_gen[idx]["generation"]
    sections = parse_sections(dataset[idx]["label"], labels_low)
    section_tokens = map_sections_into_tokens(sections, dataset_gen[idx])
    tokens = tokenize_blocksworld_generation(tokenizer, dataset_gen[idx])[0]

    for st in section_tokens:
        s = generation[st["text_pos"]:st["text_pos"]+300]
        dt = tokenizer.decode(tokens[st["pos_before"]:st["pos_before"]+300])
        inter_len = min(len(s[:200]), len(dt))
        if s[:inter_len] != dt[:inter_len]:
            print("****"*10)
            # print()
            print(tokenizer.tokenize(s[:inter_len]))
            print()
            print("----"*10)
            print(tokenizer.tokenize(dt[:inter_len]))
            print()

            # print their difference
            

test_sections()

****************************************
['So', 'Ġthe', 'Ġinitial', 'Ġstacks', 'Ġare', ':ĊĊ', 'Starting', 'Ġfrom', 'Ġthe', 'Ġtable', ':ĊĊ', 'E', 'Ġis', 'Ġon', 'Ġthe', 'Ġtable', '.', 'ĠOn', 'ĠE', 'Ġis', 'ĠF', '.', 'ĠOn', 'ĠF', 'Ġis', 'ĠC', '.', 'ĠOn', 'ĠC', 'Ġis', 'ĠB', '.', 'ĠOn', 'ĠB', 'Ġis', 'ĠA', '.', 'ĠOn', 'ĠA', 'Ġis', 'ĠD', '.', 'ĠAnd', 'ĠD', 'Ġis', 'Ġclear', '.', 'ĠSo', 'Ġthe', 'Ġstack', 'Ġis', 'ĠE', 'Ġ->', 'ĠF', 'Ġ->', 'ĠC', 'Ġ->', 'ĠB', 'Ġ->', 'ĠA', 'Ġ->', 'ĠD', '.', 'ĠAnd', 'ĠD', 'Ġis', 'Ġthe']

----------------------------------------
['Ġthe', 'Ġinitial', 'Ġstacks', 'Ġare', ':ĊĊ', 'Starting', 'Ġfrom', 'Ġthe', 'Ġtable', ':ĊĊ', 'E', 'Ġis', 'Ġon', 'Ġthe', 'Ġtable', '.', 'ĠOn', 'ĠE', 'Ġis', 'ĠF', '.', 'ĠOn', 'ĠF', 'Ġis', 'ĠC', '.', 'ĠOn', 'ĠC', 'Ġis', 'ĠB', '.', 'ĠOn', 'ĠB', 'Ġis', 'ĠA', '.', 'ĠOn', 'ĠA', 'Ġis', 'ĠD', '.', 'ĠAnd', 'ĠD', 'Ġis', 'Ġclear', '.', 'ĠSo', 'Ġthe', 'Ġstack', 'Ġis', 'ĠE', 'Ġ->', 'ĠF', 'Ġ->', 'ĠC', 'Ġ->', 'ĠB', 'Ġ->', 'ĠA', 'Ġ->', 'ĠD', '.', 'ĠAnd', 'ĠD', 

In [63]:
idx = 10

sections = parse_sections(dataset[idx]["label"], labels_low)
section_tokens = map_sections_into_tokens(sections, dataset_gen[idx])